# Gate-stream rewrites with an official-verifier fallback

A bounded, generic optimizer for the public `invsqrt` and `dqi` circuits. **Hypothesis:** conservative reset-to-reset qubit allocation and repeated-product reuse can expose savings left after the public authors' specialized rewrites. All changes are proposals: the unmodified official verifier, at its default shot counts, must return PASS and a strictly lower cost before a public component is replaced. `dfthcsos` stays byte-identical.

**Kaggle setup:** Accelerator **None / CPU**, Internet **off**, Inputs: competition **google-verifiable-quantum-circuit-compilation** + **heliosli/qcc-circuits-live**. No installs or network access. Default search and verification wall budget: 180 minutes. Four CPU cores and 30 GB RAM suffice for sequential, disk-backed passes; no full DQI token list is held in RAM.

The public portfolio is verified and packaged before search. Search failures and timeouts keep that archive. Missing inputs, a manifest mismatch or a failing public verifier stop execution: this notebook never labels an unverified portfolio ready.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TIME_BUDGET_MINUTES = 180       # whole run; 1..300, including official verification
FINAL_RESERVE_MINUTES = 45     # stop proposing changes early enough to verify + package
SCREEN_SHOTS = 4096            # randomized differential checks before each candidate verifier
SCREEN_TIMEOUT_MINUTES = 12
MAX_ROUNDS = 3
MAX_SYMBOLIC_TERMS = 8         # larger expressions become unknown, never guessed
SALT = 'flexonafft-gate-rewrites-v1'
assert 0 < TIME_BUDGET_MINUTES <= 300
assert 0 < FINAL_RESERVE_MINUTES < TIME_BUDGET_MINUTES
assert SCREEN_SHOTS >= 2048

# ── Public release and organizer reference costs ───────────────────────────────
RELEASES = {'heliosli/qcc-circuits-live': 'QCC_RELEASE_MANIFEST.json',
            'yoyobar/qcc-local-sota-20261031': 'portfolio-manifest.json'}

# Pinned bytes. A mismatch stops the notebook instead of submitting an unknown circuit.
PINNED_SHA256 = {
    'invsqrt.kmb': 'bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47',   # Ehan Mallick qf2
}

# Reference costs of the sample submission (score 1.0), as the paper's verifiers report them.
REFERENCE_COST = {
    'invsqrt': 2_076_774.233715282,          # Q 260 x C 7,987.59 (our run of the official verifier)
    'dqi': 24_439_463_583_672.93,            # Q 1,893, T ~5.725M, R(eps+) 1.1913 (as SomeJoint and busyaprime report)
    'dfthcsos': 3_355_599_588.3155365,       # lambda_eff x T x Q of the provided baseline
}
TASKS = ('invsqrt', 'dqi', 'dfthcsos')

In [ ]:
# ── Session ── paths, kickmix and the official verifiers ────────────────────────────────────────
import os, sys, glob, json, lzma, time, math, shutil, hashlib, zipfile
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
SUB = ROOT / 'submission'

def deep_find(name: str) -> list[Path]:
    """Recursive search under /kaggle/input that follows symlinked mounts (Path.rglob does not in 3.13)."""
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / name), recursive=True))

def inventory(depth: int = 3) -> str:
    """The first levels of /kaggle/input, to show what is attached."""
    lines = []
    for p in sorted(glob.glob(str(INPUT / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*' / '*'))):
        lines.append('  ' + p)
    return 'Attached inputs:\n' + ('\n'.join(lines[:60]) or '  (nothing)')

def find_competition() -> Path:
    for p in (INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation',
              INPUT / 'google-verifiable-quantum-circuit-compilation'):
        if (p / 'verifiers').is_dir():
            return p
    hits = deep_find('verify_submission_invsqrt.py')
    if hits:
        return hits[0].parent.parent.parent
    raise FileNotFoundError('Competition data not found: add the competition to the notebook inputs.\n' + inventory())

def find_release() -> tuple[str, Path]:
    for name, marker in RELEASES.items():
        hits = deep_find(marker)
        if len(hits) == 1:
            return name, hits[0].parent
        if len(hits) > 1:
            raise FileNotFoundError(f'{len(hits)} copies of {marker}: attach one version of {name}.\n' + inventory())
    raise FileNotFoundError('No release dataset found. Add one of ' + ', '.join(RELEASES)
                            + ' in Input -> Add Input.\n' + inventory())

COMP = find_competition()
RELEASE_DATASET, RELEASE = find_release()
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import numpy as np
import pandas as pd
import kickmix as km
from invsqrt import verify_submission_invsqrt as v_invsqrt
from dqi import verify_submission_dqi as v_dqi
from dfthcsos import verify as v_dfthcsos

print('competition data :', COMP)
print('release dataset  :', RELEASE_DATASET, '->', RELEASE)
print('CPU cores        :', os.cpu_count())
import re, ast, inspect, heapq, bisect, collections, multiprocessing as mp, signal, traceback
from IPython.display import display
DEADLINE = time.monotonic() + TIME_BUDGET_MINUTES * 60 - (time.time() - NB_T0)
SEARCH_DEADLINE = DEADLINE - FINAL_RESERVE_MINUTES * 60
PUBLIC = ROOT / 'public_portfolio'
EXPERIMENT = ROOT / 'gate_rewrites'; EXPERIMENT.mkdir(exist_ok=True)
LOG = []

## Recover and hash the public portfolio

Reuse the session and release extraction from `01_public_portfolio.ipynb`: symlink-following discovery, `.xz` extraction and the release manifest's SHA-256 checks. Keep immutable public files separate from the selected submission tree.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
manifest = json.loads((RELEASE / RELEASES[RELEASE_DATASET]).read_text())
if RELEASE_DATASET.startswith('heliosli/'):
    for item in manifest['files']:
        src, dst = RELEASE / item['stored_as'], SUB / item['file']
        dst.parent.mkdir(parents=True, exist_ok=True)
        if item['stored_as'].endswith('.xz'):
            with lzma.open(src) as fi, open(dst, 'wb') as fo:
                shutil.copyfileobj(fi, fo, 1 << 22)
        else:
            shutil.copyfile(src, dst)
        assert sha256(dst) == item['sha256'], f"{item['file']}: differs from the release manifest"
    print('release portfolio:', manifest.get('portfolio'), '| files matched to the manifest:', len(manifest['files']))
else:
    src = RELEASE / 'submission'
    assert src.is_dir(), f'{src} missing: dataset layout changed'
    shutil.copytree(src, SUB)

FILES = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
angles = sorted(f for f in FILES if f.startswith('dfthcsos/angle_k_'))
expected = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz', *(f'dfthcsos/angle_k_{i}.kmb' for i in range(len(angles)))}
assert set(FILES) == expected, f'unexpected submission tree: {sorted(set(FILES) ^ expected)}'

HASHES = {f: sha256(SUB / f) for f in FILES}
for name, digest in PINNED_SHA256.items():
    assert HASHES[name] == digest, f'{name}: {HASHES[name]} differs from pinned {digest}'
for f in FILES:
    print(f'{HASHES[f][:16]}  {(SUB / f).stat().st_size / 2**20:8.2f} MiB  {f}')
print(len(FILES), 'files,', len(angles), 'angle loaders; pinned hashes match')

print(json.dumps({k: v for k, v in manifest.items() if k != 'files'}, indent=1)[:3000])
if PUBLIC.exists():
    shutil.rmtree(PUBLIC)
shutil.copytree(SUB, PUBLIC)
PUBLIC_HASHES = dict(HASHES)

## Bounded verification and an archive before search

Public and candidate verifier calls use the organizer's default shot counts; no verifier code or problem specification is changed. Each expensive call runs in a disposable process group so a wall-time limit can terminate it and its workers. The selected archive is replaced atomically, only after writing and checking a complete zip. The baseline must pass before any search starts.

In [ ]:
def official(task, path):
    if task == 'invsqrt':
        circ = km.Circuit.from_file(str(path), format='kmb')
        return v_invsqrt.verify(circ, v_invsqrt.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
    if task == 'dqi':
        circ = km.Circuit.from_file(str(path), format='kmb')
        return v_dqi.verify(circ, v_dqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
    return v_dfthcsos.verify(str(path), salt=SALT)

def limited(function, args, deadline):
    """Fork avoids serializing giant circuits; children read their own paths."""
    seconds = deadline - time.monotonic()
    if seconds <= 0:
        raise TimeoutError('wall budget exhausted')
    ctx = mp.get_context('fork')
    reader, writer = ctx.Pipe(duplex=False)
    def target():
        try:
            os.setsid()
            writer.send((True, function(*args)))
        except BaseException:
            writer.send((False, traceback.format_exc()[-6000:]))
        finally:
            writer.close()
    process = ctx.Process(target=target)
    process.start(); writer.close()
    try:
        if not reader.poll(seconds):
            raise TimeoutError(f'{function.__name__} exceeded its wall budget')
        ok, value = reader.recv()
        if not ok:
            raise RuntimeError(value)
        return value
    finally:
        if process.is_alive():
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        process.join(timeout=5)
        reader.close()

def write_zip():
    paths = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
    assert paths == FILES
    tmp = ROOT / 'submission.pending.zip'
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for name in paths:
            z.write(SUB / name, name)
    with zipfile.ZipFile(tmp) as z:
        assert sorted(z.namelist()) == FILES
        assert z.testzip() is None
    tmp.replace(ROOT / 'submission.zip')

PUBLIC_RESULTS = {}
for task in TASKS:
    path = PUBLIC / (task + '.kmb' if task != 'dfthcsos' else task)
    started = time.monotonic()
    PUBLIC_RESULTS[task] = limited(official, (task, path), DEADLINE - 5 * 60)
    assert PUBLIC_RESULTS[task]['status'] == 'PASS', (task, PUBLIC_RESULTS[task])
    assert math.isfinite(float(PUBLIC_RESULTS[task]['score'])) and float(PUBLIC_RESULTS[task]['score']) > 0
    print(task, 'PUBLIC PASS', PUBLIC_RESULTS[task]['score'], f'{time.monotonic()-started:.0f}s', flush=True)
SELECTED_RESULTS = dict(PUBLIC_RESULTS)
CHOSEN = dict.fromkeys(TASKS, 'public')
CANDIDATE_RESULTS = {}
write_zip()
print('Verified public fallback archived:', ROOT / 'submission.zip')

## Parse only known instructions

Dump the actual `.kmx` prefix, gate histogram, `Circuit` statistics and simulator API at runtime. Gate lines are matched completely; comments, directives, blocks, conditions, unfamiliar syntax and unsupported operations pass through unchanged and clear symbolic knowledge. Measurements are never inserted, removed or reordered. Only exact `HMR qN bM` is treated as a zero reset. Registered qubits are fixed interface wires.

The passes stream text through files. Product reuse tracks exact bounded GF(2) expressions (no probabilistic hash equality), reuses an existing pure product wire or creates a temporary cache on a known zero wire for a bounded gate window, and uncomputes that cache before a control changes. Gate statistics are static proposal diagnostics; only the official verifier supplies executed Toffoli counts and cost.

In [ ]:
QRE = re.compile(r'(?<![A-Za-z0-9_])q(\d+)(?![A-Za-z0-9_])')
REGISTER_RE = re.compile(r'^APPEND_TO_REGISTER\s+([qb])(\d+)\s+r\d+\s*$')
ARITY = {'X':1, 'Y':1, 'Z':1, 'H':1, 'S':1, 'S_DAG':1, 'T':1, 'T_DAG':1,
         'CX':2, 'CZ':2, 'SWAP':2, 'CCX':3, 'CCZ':3, 'CSWAP':3}
SELF_INVERSE = {'X','Y','Z','H','CX','CZ','SWAP','CCX','CCZ','CSWAP'}
INVERSE = {'S':'S_DAG', 'S_DAG':'S', 'T':'T_DAG', 'T_DAG':'T'}
HMR_RE = re.compile(r'^HMR q(\d+) b(\d+)$')

def gate(line):
    parts = line.split()
    if not parts or parts[0] not in ARITY or len(parts) != ARITY[parts[0]] + 1:
        return None
    if not all(re.fullmatch(r'q\d+', v) for v in parts[1:]):
        return None
    qs = tuple(int(v[1:]) for v in parts[1:])
    if len(set(qs)) != len(qs):
        return None
    return parts[0], qs

def emit(op, qs):
    return op + ' ' + ' '.join(f'q{q}' for q in qs)

def boundary(line):
    return gate(line) is None and HMR_RE.fullmatch(line) is None

def inert_metadata(line):
    return (not line or REGISTER_RE.fullmatch(line) is not None
            or re.fullmatch(r'REGISTER r\d+ "[^"\n]*"',line) is not None
            or line.startswith('#'))

def interfaces(path):
    qubits, bits = set(), set()
    with open(path) as f:
        for line in f:
            m = REGISTER_RE.fullmatch(line.strip())
            if m:
                (qubits if m[1] == 'q' else bits).add(int(m[2]))
    if not qubits:
        raise ValueError('No registered qubit interface; refuse to guess the input layout')
    return qubits, bits

def alive(deadline):
    if time.monotonic() >= deadline:
        raise TimeoutError('search budget exhausted')

def text_stats(path):
    hist = collections.Counter(); highest = -1
    with open(path) as f:
        for i, line in enumerate(f):
            if i % 65536 == 0:
                alive(DEADLINE)
            parts = line.split()
            if parts:
                hist[parts[0]] += 1
            for q in QRE.findall(line):
                highest = max(highest, int(q))
    return {'Q_text':highest+1, 'static_Toffoli':sum(hist[k] for k in ('CCX','CCZ','CSWAP')), 'gates':dict(hist)}

def export_text(path, target):
    circuit = km.Circuit.from_file(str(path), format='kmb')
    circuit.to_file(str(target), format='kmx')
    result = {'Q':int(circuit.num_qubits), 'static_Toffoli':int(circuit.operation_statistics().num_toffoli_like)}
    del circuit
    return result

print('Simulator phase API:', [n for n in dir(km.Simulator) if 'phase' in n.lower() or 'rotat' in n.lower()])

## Width first: reset-to-reset interval colouring

Each scratch lifetime ends at an exact HMR reset. Lifetimes touching any opaque instruction or surviving to the end keep their original physical wire. Unregistered qubits start at zero, as in the simulator/verifier contract; register declarations are inert metadata. All registered wires keep their identity. Within straight-line regions, only complete zero-to-zero lifetimes can move; inclusive intervals prevent a reset and a simultaneous use sharing a wire. A greedy allocator considers future fixed intervals before borrowing their wires. It introduces no extra qubits, changes no classical bits, and leaves no cache live at a boundary. This is conservative: it cannot reduce width when the interface plus simultaneous live scratch already saturates Q.

In [ ]:
def colour_intervals(records, pinned, deadline):
    # record: logical wire, epoch, first use, final reset, fixed physical identity
    fixed = collections.defaultdict(list)
    for q, epoch, start, end, locked in records:
        if locked or q in pinned:
            fixed[q].append((start, end))
    fixed_starts = {}
    for q, spans in fixed.items():
        spans.sort(); fixed_starts[q] = [s for s, e in spans]
    def blocked(q, start, end):
        spans = fixed.get(q, ())
        if not spans:
            return False
        j = bisect.bisect_right(fixed_starts[q], end) - 1
        return j >= 0 and spans[j][1] >= start
    active, occupied, mapping = [], set(), {}
    width = max((r[0] for r in records), default=-1) + 1
    for i, (q, epoch, start, end, locked) in enumerate(sorted(records, key=lambda r:(r[2], not r[4], r[0]))):
        if i % 4096 == 0:
            alive(deadline)
        while active and active[0][0] < start:
            _, physical = heapq.heappop(active); occupied.remove(physical)
        if locked or q in pinned:
            physical = q
            assert physical not in occupied
        else:
            physical = next((p for p in range(width) if p not in pinned and p not in occupied
                             and not blocked(p, start, end)), None)
            assert physical is not None
        mapping[q, epoch] = physical
        occupied.add(physical); heapq.heappush(active, (end, physical))
    return mapping

def allocate(src, dst, deadline):
    pinned, _ = interfaces(src)
    current, records = {}, []
    unlocked = set(); seen_boundary = False
    # Non-interface qubits start at zero. Opaque control flow pins every active
    # epoch until an explicit reset proves a new zero state in a flat region.
    def state(q):
        if q not in current:
            current[q] = [0, None, seen_boundary]
            if not seen_boundary:
                unlocked.add(q)
        return current[q]
    with open(src) as f:
        for i, raw in enumerate(f):
            if i % 32768 == 0:
                alive(deadline)
            line = raw.strip()
            qs = [int(q) for q in QRE.findall(line)]
            for q in qs:
                v = state(q)
                if v[1] is None:
                    v[1] = i
            if boundary(line) and not inert_metadata(line):
                seen_boundary = True
                for q in unlocked:
                    current[q][2] = True
                unlocked.clear()
            reset = HMR_RE.fullmatch(line)
            if reset:
                q = int(reset[1]); epoch, start, locked = state(q)
                records.append((q, epoch, start, i, locked or q in pinned))
                current[q] = [epoch+1, None, False]
                unlocked.add(q)
    for q, (epoch, start, locked) in current.items():
        if start is not None:
            records.append((q, epoch, start, i+1, True))
    mapping = colour_intervals(records, pinned, deadline)
    epochs = collections.defaultdict(int)
    changed = 0
    with open(src) as fi, open(dst, 'w') as fo:
        for i, raw in enumerate(fi):
            if i % 32768 == 0:
                alive(deadline)
            line = raw.rstrip('\n')
            def replace(m):
                nonlocal changed
                q = int(m[1]); p = mapping.get((q, epochs[q]), q)
                changed += p != q
                return f'q{p}'
            fo.write(QRE.sub(replace, line) + '\n')
            reset = HMR_RE.fullmatch(line.strip())
            if reset:
                epochs[int(reset[1])] += 1
    return {'renamed_operands':changed, 'lifetimes':len(records), 'movable':sum(not r[4] for r in records)}

## Exact local gate passes

Cancel adjacent inverse pairs. Factor adjacent CCX terms with a shared target and common control: `a*b XOR a*c = a*(b XOR c)`; in a three-term quadratic form this gives a 3-to-2 AND merge (and a triangle may simplify further). Temporarily conjugate the second control with CX and restore it immediately.

A bounded symbolic pass propagates constants and exact XOR expressions, downgrades CCX/CCZ on known constant controls, and uses a live product's wire for CX/CZ copies. The extra caching pass examines small unitary windows, reserves an idle wire proved zero by a previous reset, computes an AND once, and uses CX or CZ at repeated sites before uncomputing it. Require at least four uses so the two cache CCXs strictly save Toffolis. Controls must be unchanged throughout the cache window. Unknown operations discard all knowledge. No random measurement is added.

In [ ]:
def canonical(g):
    op, q = g
    if op in ('CZ','SWAP','CCZ'):
        q = tuple(sorted(q))
    elif op == 'CCX':
        q = tuple(sorted(q[:2])) + q[2:]
    return op, q

def inverse_pair(a, b):
    if a is None or b is None:
        return False
    a, b = canonical(a), canonical(b)
    return a[1] == b[1] and ((a[0] == b[0] and a[0] in SELF_INVERSE) or INVERSE.get(a[0]) == b[0])

def factor_pair(a, b):
    if not a or not b or a[0] != 'CCX' or b[0] != 'CCX' or a[1][2] != b[1][2]:
        return None
    shared = set(a[1][:2]) & set(b[1][:2])
    if len(shared) != 1:
        return None
    c = next(iter(shared)); u = next(q for q in a[1][:2] if q != c)
    v = next(q for q in b[1][:2] if q != c); target = a[1][2]
    return [emit('CX',(v,u)), emit('CCX',(c,u,target)), emit('CX',(v,u))]

def local_rewrites(src, dst, deadline):
    pending = None; counts = collections.Counter()
    with open(src) as fi, open(dst, 'w') as fo:
        for i, raw in enumerate(fi):
            if i % 32768 == 0:
                alive(deadline)
            line = raw.strip(); g = gate(line)
            old = gate(pending) if pending is not None else None
            if inverse_pair(old, g):
                pending = None; counts['inverse_pairs'] += 1
            else:
                merged = factor_pair(old, g)
                if merged:
                    fo.write('\n'.join(merged)+'\n'); pending = None; counts['ANDs_saved'] += 1
                else:
                    if pending is not None:
                        fo.write(pending+'\n')
                    pending = line
        if pending is not None:
            fo.write(pending+'\n')
    return dict(counts)

def symbolic_rewrites(src, dst, deadline):
    values, holders, products = {}, collections.defaultdict(set), {}
    serial = 0; counts = collections.Counter()
    def fresh():
        nonlocal serial
        serial += 1
        return frozenset((serial,))
    def get(q):
        if q not in values:
            put(q, fresh())
        return values[q]
    def put(q, value):
        if q in values:
            holders[values[q]].discard(q)
        if len(value) > MAX_SYMBOLIC_TERMS:
            value = fresh()
        values[q] = value; holders[value].add(q)
    def clear():
        values.clear(); holders.clear(); products.clear()
    zero, one = frozenset(), frozenset((0,))
    def product(a, b):
        if a == zero or b == zero:
            return zero
        if a == one:
            return b
        if b == one or a == b:
            return a
        key = tuple(sorted((tuple(sorted(a)),tuple(sorted(b)))))
        if key not in products:
            products[key] = fresh()
        return products[key]
    with open(src) as fi, open(dst, 'w') as fo:
        for i, raw in enumerate(fi):
            if i % 32768 == 0:
                alive(deadline)
                clear()  # bounded memory, also forgets any previously learned zeros
            line = raw.strip(); g = gate(line); output = line
            reset = HMR_RE.fullmatch(line)
            if reset:
                put(int(reset[1]), zero)
            elif g is None:
                clear()
            else:
                op, qs = g
                if op in ('X','Y'):
                    put(qs[0], get(qs[0]) ^ one)
                elif op == 'CX':
                    a, t = qs; value = get(a)
                    if value == zero:
                        output = ''; counts['zero_CX'] += 1
                    elif value == one:
                        output = emit('X',(t,)); counts['constant_CX'] += 1
                    put(t, get(t) ^ value)
                elif op in ('CCX','CCZ'):
                    a, b, t = qs; value = product(get(a), get(b))
                    available = holders.get(value, set()) - {t}
                    if value == zero:
                        output = ''; counts['zero_Toffoli'] += 1
                    elif value == one:
                        output = emit('X' if op == 'CCX' else 'Z',(t,)); counts['constant_Toffoli'] += 1
                    elif available:
                        wire = min(available)
                        output = emit('CX' if op == 'CCX' else 'CZ',(wire,t)); counts['product_copies'] += 1
                    if op == 'CCX':
                        put(t, get(t) ^ value)
                elif op == 'CZ':
                    a,t = qs
                    if get(a) == zero or get(t) == zero:
                        output = ''; counts['zero_CZ'] += 1
                    elif get(a) == one:
                        output = emit('Z',(t,))
                    elif get(t) == one:
                        output = emit('Z',(a,))
                elif op == 'SWAP':
                    a,b = qs; va,vb = get(a),get(b); put(a,vb); put(b,va)
                elif op in ('Z','S','S_DAG','T','T_DAG'):
                    pass
                else:
                    # H and CSWAP leave the supported Boolean-expression domain.
                    for q in qs:
                        put(q, fresh())
            if output:
                fo.write(output+'\n')
    return dict(counts)

def idle_product_cache(src, dst, deadline):
    pinned, _ = interfaces(src)
    zero = set(); counts = collections.Counter(); window = []
    def flush(fo):
        if not window:
            return
        ops = [gate(line) for line in window]
        touched = {q for op, qs in ops for q in qs}
        spare = next(iter(sorted(zero - pinned - touched)), None)
        by_pair = collections.defaultdict(list)
        for j,(op,qs) in enumerate(ops):
            if op in ('CCX','CCZ'):
                by_pair[tuple(sorted(qs[:2]))].append(j)
        choice = None
        for pair, uses in by_pair.items():
            if len(uses) < 4:
                continue
            first,last = uses[0],uses[-1]
            safe = True
            for op,qs in ops[first:last+1]:
                writes = set(qs) if op in ('H','SWAP','CSWAP') else ({qs[-1]} if op in ('X','Y','CX','CCX') else set())
                if writes & set(pair):
                    safe = False; break
            if safe and (choice is None or len(uses) > len(choice[1])):
                choice = (pair,uses)
        if spare is not None and choice:
            pair,uses = choice; uses = set(uses); first,last = min(uses),max(uses)
            for j,((op,qs),line) in enumerate(zip(ops,window)):
                if j == first:
                    fo.write(emit('CCX',(*pair,spare))+'\n')
                fo.write((emit('CX' if op == 'CCX' else 'CZ',(spare,qs[-1])) if j in uses else line)+'\n')
                if j == last:
                    fo.write(emit('CCX',(*pair,spare))+'\n')
            counts['cache_chains'] += 1; counts['ANDs_saved'] += len(uses)-2
        else:
            fo.write('\n'.join(window)+'\n')
        for op,qs in ops:
            if op in ('X','Y','H','SWAP','CSWAP'):
                zero.difference_update(qs)
            elif op in ('CX','CCX'):
                zero.discard(qs[-1])
        window.clear()
    with open(src) as fi, open(dst, 'w') as fo:
        for i,raw in enumerate(fi):
            if i % 32768 == 0:
                alive(deadline)
            line = raw.strip(); g = gate(line)
            if g:
                window.append(line)
                if len(window) == 256:
                    flush(fo)
            else:
                flush(fo); fo.write(line+'\n')
                reset = HMR_RE.fullmatch(line)
                if reset:
                    zero.add(int(reset[1]))
                else:
                    zero.clear()
        flush(fo)
    return dict(counts)

## Self-checks for the rewrite algebra and allocator

These small checks exercise behavior rather than duplicating the implementation: exhaustive truth tables for the quadratic identity, unknown syntax and inverse matching, future fixed-wire conflicts, and emitted rewrites on synthetic straight-line streams. They do not require local kickmix.

In [ ]:
def truth(lines, state):
    state = dict(state)
    phase = 0
    for line in lines:
        op,q = gate(line)
        if op == 'X': state[q[0]] ^= 1
        elif op == 'CX': state[q[1]] ^= state[q[0]]
        elif op == 'CCX': state[q[2]] ^= state[q[0]] & state[q[1]]
        elif op == 'CZ': phase ^= state[q[0]] & state[q[1]]
        elif op == 'CCZ': phase ^= state[q[0]] & state[q[1]] & state[q[2]]
        elif op == 'SWAP': state[q[0]], state[q[1]] = state[q[1]], state[q[0]]
        else: raise AssertionError(op)
    return state, phase

assert gate('CCX q1 q2 q3 if b0') is None
assert gate('CX b0 q1') is None
assert inverse_pair(gate('CCX q1 q2 q3'),gate('CCX q2 q1 q3'))
original = ['CCX q0 q1 q3','CCX q0 q2 q3','CCX q1 q2 q4']
merged = factor_pair(gate(original[0]),gate(original[1])) + original[2:]
for x in range(32):
    state = {q:(x>>q)&1 for q in range(5)}
    assert truth(original,state) == truth(merged,state)
records = [(3,0,0,1,False), (2,0,2,8,False), (0,0,5,9,True), (1,0,0,9,True)]
mapping = colour_intervals(records, {1}, time.monotonic()+10)
assert mapping[2,0] not in (0,1)  # fixed q0 begins in the future
assert mapping[3,0] == 0

def rewrite_selfcheck():
    import tempfile
    with tempfile.TemporaryDirectory(dir=EXPERIMENT) as td:
        src,dst = Path(td)/'in.kmx',Path(td)/'out.kmx'
        prefix = ['REGISTER r0 "input"','APPEND_TO_REGISTER q0 r0']
        # Reset learns zero; reuse of a pure product must retain its phase action.
        gates = ['CCX q0 q1 q2','CCX q0 q1 q3','CCZ q0 q1 q4']
        src.write_text('\n'.join(prefix+['HMR q2 b0']+gates)+'\n')
        symbolic_rewrites(src,dst,time.monotonic()+10)
        body = dst.read_text().splitlines()[3:]
        assert sum(line.startswith('CC') for line in body) == 1
        for x in range(32):
            state = {q:(x>>q)&1 for q in range(5)}; state[2] = 0
            assert truth(gates,state) == truth(body,state)
        # Idle cache is zero again, with identical values and phase.
        gates = ['CCX q0 q1 q2','CCX q0 q1 q3','CCZ q0 q1 q4','CCX q0 q1 q5']
        src.write_text('\n'.join(prefix+['HMR q6 b0']+gates)+'\n')
        result = idle_product_cache(src,dst,time.monotonic()+10)
        assert result['ANDs_saved'] == 2
        body = dst.read_text().splitlines()[3:]
        for x in range(64):
            state = {q:(x>>q)&1 for q in range(7)}; state[6] = 0
            assert truth(gates,state) == truth(body,state)
        # Two zero-to-zero epochs can share a scratch wire while the interface stays fixed.
        src.write_text('\n'.join(prefix+['HMR q1 b0','HMR q2 b1','X q1','HMR q1 b2','X q2','HMR q2 b3'])+'\n')
        allocate(src,dst,time.monotonic()+10)
        assert 'APPEND_TO_REGISTER q0 r0' in dst.read_text()
        assert max(map(int,QRE.findall(dst.read_text()))) == 1  # three wires become two
        opaque = prefix+['HMR q2 b0','X q2','OPAQUE_CONTROL b0','X q2','HMR q2 b1']
        src.write_text('\n'.join(opaque)+'\n')
        allocate(src,dst,time.monotonic()+10)
        assert 'X q2' in dst.read_text()  # lifetime crossing unknown control stays fixed
rewrite_selfcheck()
print('Rewrite algebra and interval-allocation self-checks PASS')

## Randomized differential screen

For 4,096 shots, initialize the registered qubit interface with the same random basis inputs in both circuits, run `km.Simulator` with a paired seed, compare every registered quantum/classical output and all available phase readouts, and require scratch qubits to finish clean. Randomize all registered qubits rather than guessing verifier-specific register names. This strong screen can reject a valid optimization on a promise-only input domain; rejection safely keeps the public component. No shot-count overrides reach the official verifier.

Identical measurement order is an invariant of these passes. A missing phase API, simulator exception, mismatch, dirty scratch or timeout rejects the candidate. The fast screen is a filter, not a proof or a substitute for official verification.

In [ ]:
def differential_screen(public_path, candidate_path, public_text, shots=SCREEN_SHOTS):
    q_inputs,b_outputs = interfaces(public_text)
    rng = np.random.default_rng(2030321)
    q_list = sorted(q_inputs); bit_list = sorted(b_outputs)
    initial = rng.integers(0,2,(len(q_list),shots),dtype=np.uint8).astype(bool)
    initial[:,0] = False; initial[:,1] = True
    fingerprints = []
    phase_methods = [n for n in dir(km.Simulator) if n.startswith('read_')
                     and ('phase' in n.lower() or 'rotation' in n.lower()) and 'across_shots' in n]
    if not phase_methods or 'read_phase_flipped_across_shots' not in phase_methods:
        raise RuntimeError('No supported phase readout; screen refuses to guess')
    for path in (public_path,candidate_path):
        circ = km.Circuit.from_file(str(path),format='kmb')
        sim = km.Simulator(batch_size=shots,seed=481516)
        sim.write_across_shots(km.array([km.q(q) for q in q_list]),initial)
        sim.do(circ)
        q_out = np.asarray(sim.read_across_shots(km.array([km.q(q) for q in q_list]))).copy()
        bits = []
        for j in range(0,len(bit_list),512):
            bits.append(np.asarray(sim.read_across_shots(km.array([km.b(b) for b in bit_list[j:j+512]]))).copy())
        phases = {name:np.asarray(getattr(sim,name)()).copy() for name in phase_methods}
        for j in range(0,int(circ.num_qubits),512):
            scratch = [km.q(q) for q in range(j,min(j+512,int(circ.num_qubits))) if q not in q_inputs]
            if scratch and np.any(sim.read_across_shots(km.array(scratch))):
                raise AssertionError('Dirty scratch on randomized interface inputs')
        fingerprints.append((q_out,bits,phases))
        del sim,circ
    a,b = fingerprints
    assert np.array_equal(a[0],b[0]), 'registered qubit outputs differ'
    assert all(np.array_equal(x,y) for x,y in zip(a[1],b[1])), 'classical outputs differ'
    assert a[2].keys() == b[2].keys()
    for name in a[2]:
        assert np.array_equal(a[2][name],b[2][name]), f'phase differs: {name}'
    return {'status':'PASS', 'shots':shots, 'phase_readouts':phase_methods, 'seed':481516}

## Iterative search and official selection

Propose width reduction first, then local factoring/cancellation, symbolic reuse and bounded idle caching. Each pass logs Q and static Toffolis, including rejected proposals. After a cheap structural improvement test, a candidate must pass the differential screen and the full official verifier. Selection compares official cost against **both** the public component and the best accepted component, always with the same salt. Search stops before the reserve; verifier timeouts retain the last accepted portfolio. There is no change based solely on a lucky sampled cost: a proposal must also reduce Q or static Toffoli-like gates.

In [ ]:
PASSES = [('interval colouring',allocate), ('quadratic + cancellation',local_rewrites),
          ('symbolic constants + product reuse',symbolic_rewrites), ('idle AND caching',idle_product_cache)]

def search_task(task):
    public_path = PUBLIC / f'{task}.kmb'
    public_text = EXPERIMENT / f'{task}.public.kmx'
    initial = limited(export_text,(public_path,public_text),SEARCH_DEADLINE)
    with open(public_text) as f:
        print(task,'KMX prefix:\n'+''.join(next(f,'') for _ in range(24)),flush=True)
    current_text, current_path, current_stats = public_text, public_path, initial
    print(task,'gate histogram:',text_stats(public_text),flush=True)
    for round_id in range(MAX_ROUNDS):
        changed = False
        for pass_id,(name,transform) in enumerate(PASSES):
            alive(SEARCH_DEADLINE)
            stem = EXPERIMENT / f'{task}.r{round_id}.p{pass_id}'
            dst = Path(str(stem)+'.kmx'); candidate = Path(str(stem)+'.kmb')
            row = {'task':task,'round':round_id+1,'pass':name,'Q_before':current_stats['Q'],
                   'Toffoli_before':current_stats['static_Toffoli'], 'chosen':False}
            started = time.monotonic()
            try:
                detail = limited(transform,(current_text,dst,SEARCH_DEADLINE),SEARCH_DEADLINE)
                row.update(detail=detail)
                def compile_text():
                    circ = km.Circuit.from_file(str(dst),format='kmx')
                    stats = {'Q':int(circ.num_qubits),'static_Toffoli':int(circ.operation_statistics().num_toffoli_like)}
                    circ.to_file(str(candidate),format='kmb')
                    return stats
                after = limited(compile_text,(),SEARCH_DEADLINE)
                row.update(Q_after=after['Q'],Toffoli_after=after['static_Toffoli'])
                # Avoid width/Toffoli tradeoffs that merely resample an equivalent circuit.
                improved = (after['Q'] <= current_stats['Q'] and after['static_Toffoli'] <= current_stats['static_Toffoli']
                            and (after['Q'] < current_stats['Q'] or after['static_Toffoli'] < current_stats['static_Toffoli']))
                if not improved:
                    row['status'] = 'no structural improvement'
                    continue
                screen_deadline = min(SEARCH_DEADLINE,time.monotonic()+SCREEN_TIMEOUT_MINUTES*60)
                row['screen'] = limited(differential_screen,(public_path,candidate,public_text),screen_deadline)
                alive(SEARCH_DEADLINE)
                result = limited(official,(task,candidate),DEADLINE-5*60)
                cost = float(result.get('score',float('inf')))
                row.update(candidate_cost=cost,official_status=result.get('status'))
                CANDIDATE_RESULTS[task] = result
                if (result.get('status') == 'PASS' and math.isfinite(cost) and cost > 0
                    and cost < float(PUBLIC_RESULTS[task]['score'])
                    and cost < float(SELECTED_RESULTS[task]['score'])):
                    # Copy to an immutable accepted file; later proposals never overwrite it.
                    accepted = EXPERIMENT / f'{task}.accepted.r{round_id}.p{pass_id}.kmb'
                    shutil.copyfile(candidate,accepted)
                    shutil.copyfile(accepted,SUB/f'{task}.kmb')
                    SELECTED_RESULTS[task] = result; CHOSEN[task] = name
                    old_text,old_path = current_text,current_path
                    current_text,current_path,current_stats = dst,accepted,after
                    changed = True; row['chosen'] = True; row['status'] = 'accepted PASS + lower cost'
                    write_zip()
                    if old_text != public_text:
                        old_text.unlink(missing_ok=True)
                    if old_path != public_path:
                        old_path.unlink(missing_ok=True)
                else:
                    row['status'] = 'public/best retained'
            except Exception as exc:
                row['status'] = f'rejected: {type(exc).__name__}: {str(exc)[-1200:]}'
            finally:
                row['seconds'] = round(time.monotonic()-started,2); LOG.append(row)
                print(json.dumps(row,default=str),flush=True)
                # Preserve only public and accepted text; a rejected giant DQI proposal is disposable.
                if dst != current_text:
                    dst.unlink(missing_ok=True)
                candidate.unlink(missing_ok=True)
        if not changed:
            break

try:
    for task in ('invsqrt','dqi'):
        if time.monotonic() >= SEARCH_DEADLINE:
            print('Search deadline reached; retaining the verified portfolio'); break
        try:
            search_task(task)
        except Exception as exc:
            print(task,'search stopped:',type(exc).__name__,str(exc)[-1200:],flush=True)
        finally:
            # Only selected submission bytes need to survive; keep output disk use bounded.
            for temporary in EXPERIMENT.glob(f'{task}.*'):
                if temporary.suffix in ('.kmx','.kmb'):
                    temporary.unlink(missing_ok=True)
finally:
    # The baseline archive already exists. These assertions prohibit accidental portfolio drift.
    for name,digest in PUBLIC_HASHES.items():
        if name.startswith('dfthcsos/'):
            assert sha256(SUB/name) == digest
    write_zip()

if LOG:
    display(pd.DataFrame(LOG))

## Candidate versus public, provenance and submission

The ratio is organizer reference cost divided by the **chosen** component's cost. Candidate cost shows the latest officially tested proposal, even if it was rejected; a blank means none reached official verification. The expected GM is session-specific, especially for DQI's salt-dependent failure penalty. `provenance.json` contains full verifier results, the public manifest, the pass log and SHA-256 hashes for every submission member plus the final archive.

In [ ]:
rows = []
for task in TASKS:
    candidate = CANDIDATE_RESULTS.get(task,{})
    rows.append({'task':task,'public cost':float(PUBLIC_RESULTS[task]['score']),
                 'candidate cost':candidate.get('score',float('nan')),
                 'ratio':REFERENCE_COST[task]/float(SELECTED_RESULTS[task]['score']), 'chosen':CHOSEN[task]})
TABLE = pd.DataFrame(rows)
display(TABLE.style.format({'public cost':'{:.8g}','candidate cost':'{:.8g}','ratio':'{:.6f}'}))
GM = math.prod(TABLE['ratio'])**(1/3)
HASHES = {name:sha256(SUB/name) for name in FILES}
ZIP = ROOT/'submission.zip'
provenance = {'experiment':'03_gate_level_rewrites','release_dataset':RELEASE_DATASET,'release_manifest':manifest,
              'salt':SALT,'default_official_shots':True,'TIME_BUDGET_MINUTES':TIME_BUDGET_MINUTES,
              'screen_shots':SCREEN_SHOTS,'public_sha256':PUBLIC_HASHES,'sha256':HASHES,
              'submission_zip_sha256':sha256(ZIP),'public_results':PUBLIC_RESULTS,
              'candidate_results':CANDIDATE_RESULTS,'selected_results':SELECTED_RESULTS,
              'chosen':CHOSEN,'pass_log':LOG,'expected_GM':GM}
(ROOT/'provenance.json').write_text(json.dumps(provenance,indent=2,default=lambda x:x.item() if hasattr(x,'item') else str(x)))
assert all(SELECTED_RESULTS[t]['status'] == 'PASS' for t in TASKS)
assert ZIP.is_file()
print('SUBMISSION READY:',ZIP, f'| expected GM {GM:.6f}', f'| {(time.time()-NB_T0)/60:.1f} minutes')

## Expected effect

The verified public starting point is approximately **GM 23.95**. No improvement is claimed until this notebook runs on the attached circuits. At fixed Toffolis, invsqrt 62→61 qubits would multiply GM by `(62/61)^(1/3) ≈ 1.00544` (about **+0.130 GM**); DQI 892→891 would multiply it by `(892/891)^(2/3) ≈ 1.00075` (about **+0.018 GM**). A 1% DQI Toffoli reduction at unchanged failure penalty would multiply GM by `(1/0.99)^(1/3) ≈ 1.00336` (about **+0.080 GM**). These effects combine multiplicatively.

This is a residual generic search on an already optimized public base. Opaque control flow, full scratch occupancy and an exhausted time budget may leave **zero** accepted changes. Reaching GM 28 would require roughly a 1.60× gain in the product of task ratios, so the small local gains estimated here alone would not justify such a claim. The strong randomized screen may reject promise-specific but valid candidates. The public DQI decoder retains approximate rare branches; an exact rewrite preserves that behavior rather than proving the decoder exact.

# Sources and credits

The starting circuits are other participants' work from the public release dataset. Chemistry is copied byte for byte;
accepted invsqrt/DQI rewrites, if any, are explicitly recorded in provenance.json. The lineages below follow the credits the authors themselves give; if anyone is missing, please tell us in the
comments and we will add them.

- **Release datasets:** heliosli, [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live), from
  [The dataset is public now. Enjoy!](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy), whose
  assembly and re-verification this notebook follows; and yoyobar,
  [qcc-local-sota-20261031](https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031), from
  [LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits).

- **invsqrt** — Ehan Mallick, [invsqrt 20.65: two 3-to-2 AND merges on y6225](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225)
  (two GF(2) quadratic-form windows where three ANDs become two), on yoyobar and GPT-6 Astra's 62-qubit circuit.
  Earlier lineage as the authors credit it: mrmeows / team selenium
  ([21.0 circuits](https://www.kaggle.com/code/mrmeows/21-0-circuits-let-s-all-reach-the-plateau)) → Akmal Xodarev
  (busyaprime) 63 qubits → yoyobar's cone rewrites → Ahmed Berat Özer's phase correction (as distributed in manwithacat's
  [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo)); earlier still Phuc Thai (DePie),
  Dmitrii Gluzdov, heliosli and marsyas.

- **dqi** — yoyobar's selected combination of lead-product reuse and product caching on Akmal Xodarev's (busyaprime)
  [Exact DQI Rewrites at 892 Qubits](https://www.kaggle.com/code/busyaprime/lb-23-91003-exact-dqi-rewrites-at-892-qubits)
  (flag clears by X-basis measurement, symbolic AND caching, interval allocation), which uses manwithacat's two-wire
  clean ladder ([Hedge Wizardry HW-07](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)) and
  Ahmed Berat Özer's product-caching idea, on SomeJoint's (Agniv Banerjee's team)
  [892-qubit decoder](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi). SomeJoint
  credits GPT-6 Astra Pro, Claude, Ali Haydar Özdağ (flag carry), yoyobar (one-carry history) and the earlier decoder
  line: jday96314 (QuantumAI), Alex Chilton, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam and Phuc Thai (DePie).

- **dfthcsos** — Lera (marculera), [DF-THC 10x: Compact Groups From the Hamiltonian](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian):
  the (9, 143, 40) factorization, its 8-bit angle words and the loaders k = 1 to 13; Dmitrii Gluzdov,
  [Quantum Circuit Compilation from Source](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source):
  the loaders k = 0 and k = 14; heliosli: the table weights `W`, `H2_I`, continuing Ahmed Berat Özer's refinement in
  [DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression); assembly by
  manwithacat ([The Real $150K Was the Friends We Made](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made));
  Atakan Aldemir pointed heliosli to Lera's tables. Lera's own sources: Ahmed Berat Özer, somejoint, mrmeows, heliosli,
  lightningv08, kozykappa, Kozykorpesh and Sumit Chongder, Artem1410 (anazemcev), haideptry via phucthaiv02,
  Michael Rustad, k256net and Dmitrii Gluzdov.

- **Organizers:** Google Quantum AI (Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low,
  Ryan Babbush) for the tasks, the verifiers and kickmix
  ([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)), which keep their own terms.

**Our contribution:** the generic streaming parser, conservative reset-to-reset greedy interval allocator, local shared-control quadratic factoring, bounded exact symbolic propagation, idle-wire unitary product-cache proposals, differential screens, time limits and strict lower-cost selection/fallback integration. The ideas of product caching and interval allocation are credited above to busyaprime and Ahmed Berat Özer; the quadratic-form approach is credited to Ehan Mallick. We do not claim authorship of the decoder, arithmetic, factorization, or public circuit bytes. The more specialized measurement-uncomputation rewrites from the public lineage are not reimplemented here.

The public lineage includes material for which earlier authors report no explicit upstream license; preserve the original terms and provenance rather than inferring a new license for it. The circuits keep their authors' licences; our notebook code is Apache-2.0.